<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Network Addressing and Routing

The earlier access exercises showed how to open a Duckiedrone shell. For a physical Duckiedrone, that SSH connection depends on your base station finding a path across the network to the Duckiedrone. If the connection stops working, repeating the SSH command tells you little about where the problem lies.

This notebook examines the configuration behind that path and introduces network addressing and routing. You will identify a network interface and its addresses, interpret a subnet prefix, and read the routes Linux uses to send packets locally or through a gateway.

Use an authorized Duckiedrone shell opened in [Notebook 13](./13-physical-duckiedrone-ssh-access.ipynb) or [Notebook 14](./14-virtual-duckiedrone-connections.ipynb), or follow the recorded physical-Duckiedrone example below. Virtual Duckiedrones and Workspaces may show additional interfaces and different routes.

## Local networks

An SSH session to a physical Duckiedrone and a request to an internet website may leave through the same Wi-Fi interface, but follow different paths. [Figure 1](#figure-1) shows a common local network arrangement.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base station
      |
      v
    Wi-Fi access point <- Duckiedrone
      |
      v
    router
      |
      v
    provider link (modem or fiber terminal)
      |
      v
    internet
  </pre>
  <figcaption>Figure 1: A common local network arrangement.</figcaption>
</figure>

A __wireless access point__ connects Wi-Fi devices to the local network, while a __switch__ does the same for wired devices. A __router__ forwards traffic between networks. A home Wi-Fi router commonly combines all these roles in a single physical box.

To reach the internet, the router needs an upstream connection to an internet service provider (ISP). Depending on the connection technology, this may use a modem for cable or digital subscriber line (DSL), or an optical network terminal (ONT) for fiber. This equipment may be separate from the router or combined with it.

Communication between the base station and Duckiedrone on the same local network does not require that internet connection. A failed internet connection therefore does not, by itself, prevent a local SSH session.

Computers exchange data in __packets__, and Internet Protocol (__IP__) packets carry source and destination addresses. Communication between the base station and Duckiedrone can stay on the local network, while reaching an internet destination requires the router and its upstream connection.

## Interfaces and identifiers

A __network interface__ is a computer's connection to a network, such as Ethernet, Wi-Fi, or a virtual connection created by software.

Confirm your shell context with `hostname`, then inspect the interfaces:

```bash
ip -brief address
```

If `ip` is unavailable in a virtual Duckiedrone shell, do not install software just for this read-only activity. Follow the recorded physical-Duckiedrone example below, or inspect an authorized Linux environment that has `ip` and note which environment you used.

These rows were recorded on the example physical Duckiedrone `amelia`. Your Duckiedrone may have different addresses and interfaces:

```shell
lo               UNKNOWN        127.0.0.1/8 ::1/128
eth0             DOWN
wlan0            UP             192.168.1.201/24 metric 600
```

`-brief` requests compact output. The Wi-Fi interface, `wlan0`, is up and has illustrative Internet Protocol version 4 (IPv4) address `192.168.1.201`. The Ethernet interface, `eth0`, is down and has no address shown. Your addresses and interface names may differ.

`lo` is loopback: `127.0.0.1` and `::1` refer back to the current network environment. Its `UNKNOWN` state is normal here. Loopback addresses have the same values in different network environments, but refer to each environment separately.

An `UP` interface does not prove another device is reachable.

### Compare local-link and network addresses

Inspect the interface's link information:

```bash
ip -brief link
```

`amelia`'s Wi-Fi row is:

```shell
wlan0            UP             dc:a6:32:31:43:ad <BROADCAST,MULTICAST,UP,LOWER_UP>
```

The __Media Access Control (MAC) address__ identifies the interface on its local link. [Table 1](#table-1) compares its role with the other identifiers.

<table id="table-1" class="lx-table">
  <caption>Table 1: Identifiers for <code>amelia</code>'s Wi-Fi interface.</caption>
  <thead>
    <tr><th>Identifier</th><th>Value</th><th>Purpose</th></tr>
  </thead>
  <tbody>
    <tr><td>Interface name</td><td><code>wlan0</code></td><td>Select this interface in Linux</td></tr>
    <tr><td>MAC address</td><td><code>dc:a6:32:31:43:ad</code></td><td>Identify it on the local link</td></tr>
    <tr><td>IPv4 address</td><td><code>192.168.1.201</code></td><td>Address IP packets to it</td></tr>
  </tbody>
</table>

For example, network registration may require the Wi-Fi MAC address, while a program contacting the Duckiedrone uses its hostname or IP address. A MAC address is not necessarily permanent: software and Wi-Fi privacy features can change it. Review device-specific identifiers before sharing output.

## Subnets and gateways

In this example, how does the recorded Duckiedrone determine whether a destination is on its local network? Its IPv4 address has a __subnet prefix__:

```text
192.168.1.201/24
```

IPv4 addresses contain four decimal groups, each representing 8 bits. `/24` means the first 24 bits identify the network, as shown in [Figure 2](#figure-2).

<figure id="figure-2" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    192 . 168 . 1 . 201
    └───────────┘   └─┘
     network part   host part
       24 bits       8 bits
  </pre>
  <figcaption>
    Figure 2: Network and host portions of <code>amelia</code>'s IPv4 address with a /24 prefix.
  </figcaption>
</figure>

`amelia`'s subnet is therefore `192.168.1.0/24`. The equivalent __subnet mask__ is `255.255.255.0`.

Suppose the base station shares that local network and has the illustrative address `192.168.1.42/24`. [Table 2](#table-2) compares destinations against `amelia`'s prefix.

<table id="table-2" class="lx-table">
  <caption>Table 2: Example destinations relative to <code>amelia</code>'s Wi-Fi subnet.</caption>
  <thead>
    <tr><th>Destination from <code>amelia</code></th><th>Relationship to 192.168.1.0/24</th></tr>
  </thead>
  <tbody>
    <tr><td><code>192.168.1.42</code></td><td>Inside: normally reached locally</td></tr>
    <tr><td><code>192.168.2.42</code></td><td>Outside: requires an onward route</td></tr>
  </tbody>
</table>

[Figure 3](#figure-3) shows where local traffic travels in this example.

<figure id="figure-3" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base station (192.168.1.42/24)
      |
      v
    Wi-Fi access point
      |
      v
    Duckiedrone (192.168.1.201/24)
  </pre>
  <figcaption>
    Figure 3: Two hosts on the same Wi-Fi subnet.
  </figcaption>
</figure>

Comparing the first three groups works because this prefix is `/24`; it is not a rule for every prefix length.

A __gateway__ is a router used as the next step toward a destination. The __default gateway__ handles destinations without a more specific route.

## Network names

You have already used a Duckiedrone's `.local` hostname when connecting. Before sending packets, a program needs an address for that name.

__Name resolution__ supplies the address; __routing__ chooses where to send the packet. [Figure 4](#figure-4) shows these separate steps.

<figure id="figure-4" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    amelia.local
      |
      | name lookup
      v
    192.168.1.201
      |
      | route selection
      v
    outgoing network interface
  </pre>
  <figcaption>
    Figure 4: Illustrative name resolution and route selection for <code>amelia</code>.
  </figcaption>
</figure>

This mapping is illustrative: `amelia`'s interface capture alone does not verify that `amelia.local` resolves.

The Domain Name System (DNS) provides name-to-address mappings; `.local` names normally use multicast DNS (mDNS). [Notebook 21](./21-network-names-and-service-discovery.ipynb) develops these mechanisms and their connection to device discovery.

## Routing tables

Inspect the IPv4 routes:

```bash
ip route
```

These are selected entries from `amelia`'s recorded table:

```shell
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
192.168.1.1 dev wlan0 proto dhcp scope link src 192.168.1.201 metric 600
```

### Read the local route

For the example base station at `192.168.1.42`, the matching subnet route is:

```shell
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
```

[Table 3](#table-3) explains its main fields.

<table id="table-3" class="lx-table">
  <caption>Table 3: Fields in <code>amelia</code>'s local Wi-Fi route.</caption>
  <thead>
    <tr><th>Field</th><th>Meaning here</th></tr>
  </thead>
  <tbody>
    <tr><td><code>192.168.1.0/24</code></td><td>Destination subnet</td></tr>
    <tr><td><code>dev wlan0</code></td><td>Outgoing interface</td></tr>
    <tr><td><code>scope link</code></td><td>Destinations are directly connected</td></tr>
    <tr><td><code>src 192.168.1.201</code></td><td>Preferred source address for traffic <code>amelia</code> originates</td></tr>
  </tbody>
</table>

There is no `via` gateway. Traffic still passes through the Wi-Fi access point, but does not need routing into another network.

### Read the default route

For the illustrative destination `192.168.2.42`, no more specific entry matches. `amelia` uses:

```shell
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
```

[Table 4](#table-4) identifies what the default route supplies.

<table id="table-4" class="lx-table">
  <caption>Table 4: Fields in <code>amelia</code>'s default route.</caption>
  <thead>
    <tr><th>Field</th><th>Meaning here</th></tr>
  </thead>
  <tbody>
    <tr><td><code>default</code></td><td>Destinations without a more specific match</td></tr>
    <tr><td><code>via 192.168.1.1</code></td><td>Next-hop gateway on the local network</td></tr>
    <tr><td><code>dev wlan0</code></td><td>Outgoing interface</td></tr>
    <tr><td><code>src 192.168.1.201</code></td><td>Preferred source address for traffic <code>amelia</code> originates</td></tr>
  </tbody>
</table>

The first hop is gateway `192.168.1.1`, reached through `wlan0`, as shown in [Figure 5](#figure-5).

<figure id="figure-5" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    Duckiedrone (192.168.1.201)
      |
      v
    gateway (192.168.1.1)
      |
      ?
      |
      v
    destination (192.168.2.42)
  </pre>
  <figcaption>
    Figure 5: The default route identifies the first hop, not the entire path.
  </figcaption>
</figure>

The question mark matters: `amelia`'s table does not establish whether the gateway can forward the packet successfully.

### Understand which entry wins

For ordinary routing in this table, the __most specific matching destination prefix__ wins, which is not necessarily in the order lines are printed. [Table 5](#table-5) shows the selected entries.

<table id="table-5" class="lx-table">
  <caption>Table 5: Route selection examples from <code>amelia</code>'s table.</caption>
  <thead>
    <tr><th>Destination</th><th>Selected entry</th></tr>
  </thead>
  <tbody>
    <tr><td><code>192.168.1.42</code></td><td>Local subnet <code>192.168.1.0/24</code></td></tr>
    <tr><td><code>192.168.2.42</code></td><td><code>default</code></td></tr>
    <tr><td><code>192.168.1.1</code></td><td>Route specifically to <code>192.168.1.1</code></td></tr>
  </tbody>
</table>

`proto kernel` identifies a kernel-installed route. `proto dhcp` identifies one supplied through automatic configuration using Dynamic Host Configuration Protocol (DHCP). A `metric` helps choose between otherwise comparable routes; lower values are preferred. See the [`ip route` manual](https://man7.org/linux/man-pages/man8/ip-route.8.html).

### Try it

Using your own output, identify:

1. A non-loopback interface with an IP address and prefix.

2. The route for a directly connected subnet.

3. The default gateway, if one is configured.

Compare these with `amelia`'s example. Virtual environments may have several software-created interfaces; their names and addresses need not match.

__Question:__ If the default route disappeared but the Wi-Fi subnet route remained, would `amelia` still have a route to `192.168.1.42`?

<details>
<summary>Check your result</summary>

Yes. The local `192.168.1.0/24` entry still covers that address. Destinations that depended on the default route would lose their route. The inspected routes describe configuration, not successful delivery.

</details>

[Notebook 22](./22-network-diagnostics-and-testing.ipynb) adds connection tests.

## Address families together

`amelia`'s capture also contains `::1`, an __Internet Protocol version 6 (IPv6)__ address. IPv6 uses 128-bit addresses, compared with IPv4's 32 bits.

IPv6 addresses use hexadecimal groups separated by colons. [Figure 6](#figure-6) shows how one consecutive run of zero groups can be shortened to `::`.

<figure id="figure-6" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    2001:db8:0:0:0:0:0:42
      |
      v
    2001:db8::42
  </pre>
  <figcaption>
    Figure 6: Compressing a run of zero groups in an IPv6 address.
  </figcaption>
</figure>

This address is reserved for documentation. [Table 6](#table-6) lists two forms to recognize in real output.

<table id="table-6" class="lx-table">
  <caption>Table 6: Useful IPv6 address forms.</caption>
  <thead>
    <tr><th>Address</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>::1</code></td><td>IPv6 loopback in the current network environment</td></tr>
    <tr><td>An address beginning <code>fe80::</code></td><td>A common link-local form; routers do not forward it beyond the local link</td></tr>
  </tbody>
</table>

These forms are defined in the [IPv6 addressing specification](https://www.rfc-editor.org/rfc/rfc4291.html).

A computer can use IPv4 and IPv6 together, called __dual stack__, but success with one does not prove the other works. `amelia`'s capture shows IPv6 loopback and no IPv6 address on `wlan0`. Likewise, a link-local IPv6 address alone does not establish IPv6 internet access.

## Further reading

The Linux [`ip address` manual](https://man7.org/linux/man-pages/man8/ip-address.8.html) describes assigned addresses and interface information.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [1]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
